<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 3 — BM25 y evaluación de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Mejor ranking que TF-IDF, y cómo medirlo con métricas de IR</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Dos partes. **A)** Implementar BM25 desde cero y comparar su ranking contra el motor TF-IDF del Lab 2. **B)** Construir juicios de relevancia (qrels) y medir ambos sistemas con métricas de IR para decidir, con números, cuál es mejor.


## 0 · Corpus procesado del Lab 1

In [16]:
import json, math, re, unicodedata
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Línea base: su motor TF-IDF del Lab 2
Necesitan el buscador TF-IDF como punto de comparación. Peguen sus funciones del Lab 2.

In [17]:
import json
import math
import re
import unicodedata
from collections import Counter
import operator
import spacy

# 1. Configuración del entorno
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'ni', 'sin'} 
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

# 2. Cargar corpus
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos cargados.')

# 3. Pipeline de preprocesamiento
def normalizar(texto, quitar_acentos=True):
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    if quitar_acentos:
        texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', texto)).strip()

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    return [t.lemma_ for t in doc if t.is_alpha and not t.is_space and t.text not in MIS_STOPWORDS]

# 4. Motor TF-IDF (Lab 2) desde cero
def tf(doc):
    if not doc: return {}
    c = Counter(doc)
    n = len(doc)
    return {t: f / n for t, f in c.items()}

def idf(corpus_docs):
    N = len(corpus_docs)
    df_conteo = Counter()
    for doc in corpus_docs:
        for t in set(doc): df_conteo[t] += 1
    return {t: math.log(N / df_t) for t, df_t in df_conteo.items()}

def tfidf(doc, idf_dict):
    vector_tf = tf(doc)
    return {t: f * idf_dict.get(t, 0.0) for t, f in vector_tf.items()}

def coseno(v1, v2):
    producto_punto = sum(v1[t] * v2[t] for t in v1 if t in v2)
    norma_v1 = math.sqrt(sum(w**2 for w in v1.values()))
    norma_v2 = math.sqrt(sum(w**2 for w in v2.values()))
    if norma_v1 == 0.0 or norma_v2 == 0.0: return 0.0
    return producto_punto / (norma_v1 * norma_v2)

IDF_TFIDF = idf(documentos)
INDICE_TFIDF = [tfidf(doc, IDF_TFIDF) for doc in documentos]

def vectorizar_consulta(texto):
    tokens_q = preprocesar(texto)
    tf_q = tf(tokens_q)
    return {t: f * IDF_TFIDF.get(t, 0.0) for t, f in tf_q.items() if t in IDF_TFIDF}

def buscar_tfidf(consulta, k=5):
    q_vec = vectorizar_consulta(consulta)
    resultados = []
    for i, doc_vec in enumerate(INDICE_TFIDF):
        score = coseno(q_vec, doc_vec)
        resultados.append((corpus[i]['id'], corpus[i]['titulo'], score))
    return sorted(resultados, key=lambda x: x[2], reverse=True)[:k]

14 documentos cargados.


---
## Parte A · BM25 desde cero

**A.1** IDF de BM25 (variante suavizada, nunca negativa) y la función `bm25`. Sigan la fórmula de la clase:
$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f\,(k_1+1)}{f+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

In [18]:
# Longitud media de los documentos en el corpus
avgdl = sum(len(doc) for doc in documentos) / len(documentos)
print(f"Longitud promedio de documentos (avgdl): {avgdl:.2f}")

def idf_bm25(corpus_docs):
    """Calcula el IDF suavizado de BM25 para evitar valores negativos."""
    N = len(corpus_docs)
    df_conteo = Counter()
    for doc in corpus_docs:
        for t in set(doc):
            df_conteo[t] += 1
            
    idf_dict = {}
    for t, df_t in df_conteo.items():
        # Fórmula Robertson-Spärck Jones suavizada
        idf_dict[t] = math.log(1.0 + (N - df_t + 0.5) / (df_t + 0.5))
    return idf_dict

IDF_BM25 = idf_bm25(documentos)

def bm25(doc, q_tokens, k1=1.5, b=0.75):
    """Calcula el score de BM25 acumulado para los tokens de la consulta."""
    score = 0.0
    doc_counts = Counter(doc)
    dl = len(doc)
    
    for t in q_tokens:
        if t in IDF_BM25:
            f = doc_counts.get(t, 0)
            if f > 0:
                numerador = f * (k1 + 1.0)
                denominador = f + k1 * (1.0 - b + b * (dl / avgdl))
                score += IDF_BM25[t] * (numerador / denominador)
    return score

Longitud promedio de documentos (avgdl): 16.29


**A.2** Buscador BM25, análogo a `buscar_tfidf` pero ranqueando por score BM25.

In [19]:
def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    q_tokens = preprocesar(consulta)
    resultados = []
    for i, doc in enumerate(documentos):
        score = bm25(doc, q_tokens, k1=k1, b=b)
        resultados.append((corpus[i]['id'], corpus[i]['titulo'], score))
    return sorted(resultados, key=lambda x: x[2], reverse=True)[:k]

**A.3** Comparación lado a lado. Para 3 consultas, muestren el top-5 de TF-IDF y de BM25 en paralelo y marquen qué cambió.

In [20]:
# Consultas para comparar los modelos
consultas_test = [
    'sequia y cultivos de maiz',
    'problemas de agua potable',
    'produccion artesanal de cafe y cacao'
]

print(f"{'TF-IDF Ranking':<38} | {'BM25 Ranking':<38}")
print("-" * 80)
for q in consultas_test:
    print(f"Consulta: '{q}'")
    res_tfidf = buscar_tfidf(q, k=3)
    res_bm25 = buscar_bm25(q, k=3)
    
    for idx in range(3):
        t_id = res_tfidf[idx][0] if idx < len(res_tfidf) else "-"
        b_id = res_bm25[idx][0] if idx < len(res_bm25) else "-"
        t_sc = f"{res_tfidf[idx][2]:.3f}" if idx < len(res_tfidf) else "-"
        b_sc = f"{res_bm25[idx][2]:.3f}" if idx < len(res_bm25) else "-"
        
        cambio = "(*)" if t_id != b_id else "   "
        print(f"  {idx+1}. [{t_id}] ({t_sc}) {cambio}           |   {idx+1}. [{b_id}] ({b_sc})")
    print("=" * 80)

TF-IDF Ranking                         | BM25 Ranking                          
--------------------------------------------------------------------------------
Consulta: 'sequia y cultivos de maiz'
  1. [d04] (0.447)               |   1. [d04] (6.633)
  2. [d02] (0.083)               |   2. [d02] (1.667)
  3. [d01] (0.000)               |   3. [d01] (0.000)
Consulta: 'problemas de agua potable'
  1. [d13] (0.505)               |   1. [d13] (5.502)
  2. [d01] (0.000)               |   2. [d01] (0.000)
  3. [d02] (0.000)               |   3. [d02] (0.000)
Consulta: 'produccion artesanal de cafe y cacao'
  1. [d08] (0.347)               |   1. [d08] (5.933)
  2. [d12] (0.180)               |   2. [d12] (3.612)
  3. [d03] (0.090)               |   3. [d03] (1.806)


**A.4** Barrido de parámetros. Observen cómo se mueve el ranking de una consulta al variar (k₁, b).

In [21]:
consulta_barrido = 'produccion artesanal de cafe y cacao'
valores_k1 = [1.2, 2.0]
valores_b = [0.0, 0.75, 1.0]

print(f"Barrido de parámetros para la consulta: '{consulta_barrido}'\n")
for k1 in valores_k1:
    for b in valores_b:
        top_3 = buscar_bm25(consulta_barrido, k=3, k1=k1, b=b)
        ranking_str = ", ".join([f"{doc_id}({sc:.2f})" for doc_id, _, sc in top_3])
        print(f"Parámetros -> k1={k1:<3} b={b:<4} | Top-3: [{ranking_str}]")

Barrido de parámetros para la consulta: 'produccion artesanal de cafe y cacao'

Parámetros -> k1=1.2 b=0.0  | Top-3: [d08(5.89), d12(3.58), d03(1.79)]
Parámetros -> k1=1.2 b=0.75 | Top-3: [d08(5.93), d12(3.61), d03(1.80)]
Parámetros -> k1=1.2 b=1.0  | Top-3: [d08(5.94), d12(3.62), d03(1.81)]
Parámetros -> k1=2.0 b=0.0  | Top-3: [d08(5.89), d12(3.58), d03(1.79)]
Parámetros -> k1=2.0 b=0.75 | Top-3: [d08(5.94), d12(3.62), d03(1.81)]
Parámetros -> k1=2.0 b=1.0  | Top-3: [d08(5.96), d12(3.63), d03(1.81)]


En nuestro corpus, el orden cualitativo de los documentos se mantuvo estable (d08, d12, d03), pero los scores numéricos variaron de forma consistente.Efecto de $b$ (Normalización por longitud): Al mover $b$ de 0.0 a 1.0, los scores de los tres documentos se incrementaron de manera gradual (por ejemplo, d08 subió de 5.89 a 5.94). Esto ocurre porque la longitud promedio del corpus (avgdl = 16.29) es mayor que la longitud específica de estos documentos altamente relevantes (que son cortos y concisos). Al activar la penalización por longitud ($b > 0$), el algoritmo premia la densidad de información disminuyendo el denominador de los documentos más cortos que el promedio, lo que eleva su score.Efecto de $k_1$ (Saturación de TF): Al incrementar $k_1$ de 1.2 a 2.0, los scores volvieron a subir en combinación con $b=0.75$ y $b=1.0$. Un valor de $k_1$ más alto expande el rango en el cual las apariciones adicionales de un término siguen aportando peso antes de saturarse. En documentos cortos donde un término clave aparece pocas veces, elevar $k_1$ estira la curva de ganancia de frecuencia local, incrementando el score final.

---
## Parte B · Evaluación con métricas de IR

**B.1** Juicios de relevancia (qrels). Etiqueten a mano, con relevancia graduada (0–3), los documentos relevantes de **5 consultas** sobre su corpus. Completen el diccionario.

In [22]:
qrels = {
    'sequia y cultivos de maiz': {'d04': 3, 'd02': 2},
    'problemas de agua potable': {'d13': 3, 'd02': 1, 'd01': 1},
    'produccion artesanal de cafe y cacao': {'d03': 3, 'd08': 3, 'd12': 2},
    'infraestructura carretera y seguridad': {'d10': 3},
    'casos de enfermedades y dengue': {'d11': 3}
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'
print("Juicios de relevancia guardados y verificados con éxito.")

Juicios de relevancia guardados y verificados con éxito.


**B.2** Métricas desde cero: `precision_at_k`, `recall_at_k`, `mrr`, `average_precision` y `ndcg_at_k`.

In [23]:
def _rel(qid, doc): 
    return qrels[qid].get(doc, 0)

def _relevantes(qid): 
    return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    if not top_k: return 0.0
    hits = sum(1 for doc in top_k if _rel(qid, doc) > 0)
    return hits / k

def recall_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    rel_totales = _relevantes(qid)
    if not rel_totales: return 0.0
    hits = sum(1 for doc in top_k if _rel(qid, doc) > 0)
    return hits / len(rel_totales)

def mrr(ranking, qid):
    for idx, doc in enumerate(ranking):
        if _rel(qid, doc) > 0:
            return 1.0 / (idx + 1)
    return 0.0

def average_precision(ranking, qid):
    rel_totales = _relevantes(qid)
    if not rel_totales: return 0.0
    
    suma_precisiones = 0.0
    hits = 0
    for idx, doc in enumerate(ranking):
        if _rel(qid, doc) > 0:
            hits += 1
            suma_precisiones += hits / (idx + 1)
            
    return suma_precisiones / len(rel_totales)

def ndcg_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    
    # Calcular DCG@k
    dcg = 0.0
    for idx, doc in enumerate(top_k):
        r = _rel(qid, doc)
        dcg += (2**r - 1) / math.log2(idx + 2)
        
    # Calcular IDCG@k (Ranking ideal ordenando los qrels reales de forma decreciente)
    ganancias_ideales = sorted([g for d, g in qrels[qid].items()], reverse=True)[:k]
    idcg = 0.0
    for idx, g in enumerate(ganancias_ideales):
        idcg += (2**g - 1) / math.log2(idx + 2)
        
    if idcg == 0.0: return 0.0
    return dcg / idcg

**B.3** Evalúen ambos sistemas sobre las 5 consultas y promedien cada métrica.

In [24]:
def evaluar(buscar_fn, **kwargs):
    tot_p5, tot_r5, tot_mrr, tot_ap, tot_ndcg5 = 0.0, 0.0, 0.0, 0.0, 0.0
    n_q = len(qrels)
    
    for qid in qrels.keys():
        # Extraer únicamente el string de IDs del ranking retornado
        ranking_ids = [res[0] for res in buscar_fn(qid, k=14, **kwargs)]
        
        tot_p5 += precision_at_k(ranking_ids, qid, k=5)
        tot_r5 += recall_at_k(ranking_ids, qid, k=5)
        tot_mrr += mrr(ranking_ids, qid)
        tot_ap += average_precision(ranking_ids, qid)
        tot_ndcg5 += ndcg_at_k(ranking_ids, qid, k=5)
        
    return {
        'P@5': tot_p5 / n_q, 'R@5': tot_r5 / n_q, 'MRR': tot_mrr / n_q,
        'MAP': tot_ap / n_q, 'nDCG@5': tot_ndcg5 / n_q
    }

metrics_tfidf = evaluar(buscar_tfidf)
metrics_bm25 = evaluar(buscar_bm25, k1=1.5, b=0.75)

print(f"{'Métrica':<10} | {'TF-IDF':<10} | {'BM25':<10} | {'Mejora':<10}")
print("-" * 48)
for m in metrics_tfidf.keys():
    v_tf = metrics_tfidf[m]
    v_bm = metrics_bm25[m]
    diff = v_bm - v_tf
    print(f"{m:<10} | {v_tf:<10.3f} | {v_bm:<10.3f} | {diff:+=10.3f}")

Métrica    | TF-IDF     | BM25       | Mejora    
------------------------------------------------
P@5        | 0.400      | 0.400      | +++++0.000
R@5        | 1.000      | 1.000      | +++++0.000
MRR        | 1.000      | 1.000      | +++++0.000
MAP        | 1.000      | 1.000      | +++++0.000
nDCG@5     | 0.992      | 0.992      | +++++0.000


**B.4** ¿Qué sistema y qué (k₁, b) eligen para producción, y **con qué métrica** lo justifican?

_Su decisión:_ Elegimos el sistema BM25 configurado con los parámetros estándar de la industria ($k_1 = 1.5, b = 0.75$).
_Justificación:_ Al observar la tabla comparativa, ambos sistemas obtienen métricas perfectas (MAP = 1.000, MRR = 1.000) y un nDCG@5 idéntico de 0.992. Esta falta de variación matemática (mejora de +0.000) se debe a que estamos evaluando un corpus de juguete muy pequeño (14 documentos) y consultas sumamente dirigidas, donde los pocos documentos relevantes son devueltos de forma exacta en el Top-1 por ambos motores, impidiendo ver diferencias en los agregados estadísticos.Sin embargo, elegimos BM25 con nDCG@5 como métrica de justificación estratégica. En un entorno de producción real con miles de documentos, TF-IDF lineal/coseno comete el error de indexar prioritariamente textos largos redundantes (verbosidad) y sobre-ponderar la repetición artificial de palabras clave. BM25 mitiga esto gracias a que amortigua el término de frecuencia local ($k_1$) y penaliza los textos inflados ($b$). Dado que nuestros qrels manejan relevancia graduada (donde un acierto 3 vale mucho más que un 1), nDCG@5 es la métrica idónea para producción, y BM25 garantiza teóricamente el mejor ordenamiento decreciente de esas ganancias.

## Entregables — Lab 3
- [ ] `idf_bm25`, `bm25`, `buscar_bm25` desde cero + comparación y barrido (Parte A).
- [ ] `qrels` de 5 consultas + las 5 métricas desde cero (Parte B).
- [ ] Tabla comparativa TF-IDF vs BM25 y **decisión justificada con una métrica**.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
